In [ ]:
#!pip install weaviate-client --upgrade --no-deps
#!pip install -U weaviate-client

# Project Setup

## Overview
This notebook is part of a reproducible GRPO-based fine-tuning pipeline for cybersecurity policy generation.

## Purpose of this setup cell
The first code cell in this notebook:
- loads environment variables from `.env`
- identifies the project root directory automatically
- defines standard folder paths used across the project
- creates required folders if they do not already exist

## Why this matters
This makes the notebook portable across macOS, Windows, and Linux without requiring users to manually edit file paths.

## Expected project folders
- `data/corpus/` → source PDF corpus
- `data/processed/` → intermediate processed data
- `data/sample/` → optional small example data
- `outputs/completions/` → generated completions
- `outputs/rankings/` → ranked outputs
- `outputs/models/` → trained model checkpoints
- `outputs/evaluations/` → evaluation results

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# Define PROJECT_ROOT automatically
if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

# Define folders
DATA_DIR = PROJECT_ROOT / "data"
CORPUS_DIR = DATA_DIR / "corpus"
PROCESSED_DIR = DATA_DIR / "processed"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
COMPLETIONS_DIR = OUTPUT_DIR / "completions"
RANKINGS_DIR = OUTPUT_DIR / "rankings"
MODELS_DIR = OUTPUT_DIR / "models"
EVAL_DIR = OUTPUT_DIR / "evaluations"

# Create folders if needed
for folder in [
    DATA_DIR, CORPUS_DIR, PROCESSED_DIR,
    OUTPUT_DIR, COMPLETIONS_DIR, RANKINGS_DIR, MODELS_DIR, EVAL_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)

# 02 Step 2 — Generate Multiple Policy Completions

## Purpose
This notebook generates multiple candidate cybersecurity policy completions for each prompt using the selected LLaMA model.

## Why this step is important
GRPO training requires groups of candidate outputs rather than a single answer. These multiple completions form the comparison set that will later be judged and ranked.

## Inputs
- Retrieved cybersecurity context from Weaviate
- Model access credentials from `.env`

## Outputs
- A JSONL file containing prompts and multiple generated completions by the base model saved to `outputs/completions/`

## Main tasks
1. Load environment variables and authenticate model access
2. Connect to Weaviate to retrieve context
3. Build enriched prompts
4. Generate multiple completions per prompt
5. Save the generated outputs as JSONL

## Success criteria
This step is complete when each prompt has multiple generated completions saved in the completions output folder.

In [2]:
import pandas as pd
from huggingface_hub import login
from transformers import AutoTokenizer
import weaviate
import os, weaviate
from weaviate.classes.init import Auth
from sentence_transformers import SentenceTransformer

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [ ]:
# Configurations
DATASET_SIZE = 300
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

HF_TOKEN = os.getenv("HF_TOKEN")
login(HF_TOKEN)


In [4]:

from huggingface_hub import InferenceClient
from huggingface_hub.utils import HfHubHTTPError
import os, time, random


def _make_client(model=MODEL_ID, provider=None, token=HF_TOKEN, timeout=180):
    return InferenceClient(model=model, provider=provider, token=token, timeout=timeout)

_CLIENT = _make_client(provider=None)  # start on HF default endpoint

def complete_chat(system_prompt, user_prompt, model=MODEL_ID, provider=None,
                  max_tokens=600, temperature=0.7, top_p=0.95,
                  max_retries=8):
    global _CLIENT
    if (model != MODEL_ID) or (provider is not None):
        _CLIENT = _make_client(model=model, provider=provider)

    messages = [
  {"role":"system","content":
   "Write a full cybersecurity policy for a medium company. At least 1,800 words. "
   "Use sections: Purpose, Scope, Roles, Asset Mgmt, Data Classification, Access Control, "
   "Authentication/MFA, Endpoint Security, Network Security, App/Secure Dev, "
   "Vuln Mgmt & Patching, Logging & Monitoring, Backup/BCP/DR, Incident Response, "
   "Third-Party/Supply Chain, Training & Awareness, Exceptions, Review & Governance. "
   "End with the marker <<END>>."
  },
  {"role":"user","content": "Company: [Name], 250 employees, SaaS, PII + PHI, ISO 27001 + NIST CSF, Canada/US, GDPR/HIPAA."}
]

    attempt = 0
    local_max_tokens = max_tokens
    local_top_p = top_p
    while True:
        try:
            resp = _CLIENT.chat_completion(
                messages=messages,
                max_tokens=local_max_tokens,
                temperature=temperature,
                top_p=local_top_p
            )
            return resp.choices[0].message["content"]
        except HfHubHTTPError as e:
            status = getattr(e.response, "status_code", None)
            # backoff for 429/5xx including 504
            if status in (429,) or (status and 500 <= status <= 599):
                wait = min(1.4 * (2 ** attempt) + random.uniform(0, 0.7), 45)
                # tighten request a bit on 5xx
                if status and 500 <= status <= 599:
                    local_max_tokens = max(256, int(local_max_tokens * 0.85))
                    local_top_p      = max(0.80, local_top_p - 0.03)
                time.sleep(wait); attempt += 1
                if attempt > max_retries:
                    raise
            else:
                raise
        except Exception as e:
            wait = min(1.4 * (2 ** attempt) + random.uniform(0, 0.7), 45)
            time.sleep(wait); attempt += 1
            if attempt > max_retries:
                raise RuntimeError(f"chat_completion failed after retries: {e}") from e




In [ ]:
import weaviate
from weaviate.auth import AuthApiKey
import os
# Create a Weaviate Client for RAG and Prompt Augmentation

WEAVIATE_URL= os.environ["WEAVIATE_URL"] 
WEAVIATE_API_KEY = os.environ["WEAVIATE_API_KEY"] 

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=AuthApiKey(WEAVIATE_API_KEY)
)
if client.is_ready():
    print("Weaviate is connected and ready!")
else:
    print("Weaviate connection failed. Check your URL or API key.")
col = client.collections.get("CyberPolicyChunk")



Weaviate is connected and ready!


In [6]:
# System prompt
system_prompt = """
You are an AI assistant helping to generate cybsecurity policies for companies. To generate the cybersecurity policy, you are provided with the description of the user's company as well as chunks of existing cybsecurity policies which are similar to the user's query. Use the information in the query, the provided cybersecurity policy chunks, and your own internal knowledge to generate a cybersecurity policy for the user.
"""

In [7]:
# User prompt
geographical_presence_options = [
    "Europe (EU countries, UK)",              # General European region with regulatory specifics like GDPR
    "Asia-Pacific (China, India, Australia)", # Major Asia-Pacific countries with unique security concerns
    "Middle East & Africa",                   # Middle East and African countries with diverse regulatory and security needs
    "South America (Brazil, Argentina)",      # Key countries in South America
    "North America (USA, Canada, Mexico)",    # North America with a focus on regional differences in cybersecurity
    "Central Asia (Kazakhstan, Uzbekistan)",  # Central Asian countries with evolving cybersecurity practices
    "Southeast Asia (Thailand, Vietnam, Philippines)", # Southeast Asian nations with growing tech adoption
    "Nordic countries (Sweden, Norway, Finland)", # High-standard data protection and cybersecurity practices
    "Eastern Europe (Poland, Romania, Hungary)", # Emerging cybersecurity frameworks in Eastern European nations
    "Western Europe (France, Germany, Netherlands)", # Developed economies with strong cybersecurity laws
    "Latin America (Chile, Colombia, Peru)",  # Additional South American context
    "Caribbean (Jamaica, Trinidad and Tobago)", # Caribbean nations with regional cybersecurity concerns
    "Sub-Saharan Africa (South Africa, Nigeria, Kenya)", # Key African countries with unique cybersecurity needs
    "Middle Asia (Georgia, Armenia)",          # Smaller but important countries in the region
    "East Asia (Japan, South Korea, Taiwan)",  # Developed economies with high tech adoption and security focus
    "Southern Europe (Italy, Spain, Greece)",  # Countries with specific compliance challenges and regulations
    "Northern Europe (Denmark, Ireland, Estonia)", # Advanced cybersecurity practices in Northern European nations
]
company_size_options = ["", "", "", "small", "small (less than 10)", "medium", "medium (10 to 50)", "large", "large (50 or more)"]
industry_sector_options = [
    "technology",               # General tech industry
    "financial",                # Banking, investment, and financial services
    "healthcare",               # Hospitals, clinics, and healthcare providers
    "law",                      # Legal services and law firms
    "manufacturing",            # Factories and industrial manufacturing
    "retail",                   # Retail businesses, both online and brick-and-mortar
    "education",                # Educational institutions and e-learning platforms
    "government",               # Public sector and government agencies
    "energy",                   # Energy production, distribution, and utilities
    "transportation",           # Airlines, logistics, and public transport
    "real estate",              # Real estate agencies and property management
    "telecommunications",       # Communication services and telecom companies
    "pharmaceutical",          # Drug manufacturers and related industries
    "automotive",               # Automotive manufacturers and suppliers
    "hospitality",              # Hotels, restaurants, and travel services
    "construction",             # Construction companies and building services
    "media and entertainment",  # Film, television, music, and media platforms
    "non-profit",               # Non-profit organizations and foundations
    "insurance",                # Insurance companies and brokers
    "legal technology",         # Tech companies offering legal services solutions
    "cybersecurity",            # Companies specializing in cybersecurity solutions
    "agriculture",              # Agriculture and food production
    "mining",                   # Mining and extraction industries
    "aerospace",                # Aerospace manufacturers and related industries
    "financial technology (FinTech)", # FinTech startups and tech-based financial solutions
    "logistics",                # Supply chain management and logistics services
    "data centers",             # Companies managing data storage and processing
    "software development",     # Custom software and app development firms
    "consulting",               # Management and strategy consulting firms
    "gaming",                   # Video game development and gaming companies
    "e-commerce",               # Online shopping and digital marketplaces
    "social media",             # Social networking platforms
    "biotechnology",            # Biotech companies focused on research and development
    "space exploration",        # Space agencies and companies involved in space missions
    "sports",                   # Sports teams, leagues, and athletic organizations
]
third_party_dependency_options = [
    "vendors",                         # General service providers
    "contractors",                     # Independent workers or specialists
    "partners in common industries",   # Strategic business partners in similar fields
    "cloud service providers",         # Providers of cloud storage and computing services
    "consultants",                     # External experts for security and compliance
    "outsourced IT services",          # External IT management and support teams
    "software as a service (SaaS) providers", # Companies that provide cloud-based applications
    "data processors",                 # Entities that process data on behalf of the data controller
    "third-party auditors",            # Independent audit firms for security assessments
    "logistics partners",              # Companies involved in logistics and distribution that handle data
    "contract manufacturers",          # Companies that manufacture products under contract
    "subcontractors",                  # Sub-level contractors working under primary contractors
    "business affiliates",             # Companies sharing a business relationship or network
    "marketing agencies",              # External agencies involved in data-driven marketing
    "payment processors",              # Companies handling payment processing and related services
    "external consultants",            # Consultants hired for specialized security reviews
    "freelance developers",            # Independent developers working on projects
    "resellers",                       # Companies that resell products or services
    "data sharing partners",           # Partners involved in data sharing agreements
    "research collaborators",          # Partners in collaborative research projects
    "regulatory bodies",               # Entities that oversee industry regulations
    "insurance providers",             # Companies providing cyber insurance coverage
    "technology partners",             # Companies providing technology solutions or integrations
    "law enforcement",                 # Law agencies collaborating in case of cyber incidents
    "cloud storage vendors"            # Providers offering data storage solutions
]
regulations_options = [
    "HIPAA",               # Health Insurance Portability and Accountability Act
    "PCI-DSS",             # Payment Card Industry Data Security Standard
    "GDPR",                # General Data Protection Regulation
    "CCPA",                # California Consumer Privacy Act
    "FISMA",               # Federal Information Security Management Act
    "SOX",                 # Sarbanes-Oxley Act
    "NIST SP 800-53",      # Security and Privacy Controls for Federal Information Systems
    "CISA",                # Cybersecurity Information Sharing Act
    "COPPA",               # Children's Online Privacy Protection Act
    "GLBA",                # Gramm-Leach-Bliley Act
    "FERPA",               # Family Educational Rights and Privacy Act
    "ISO 27001",           # International standard for information security management
    "SOC 2",               # Service Organization Control 2 reports for data protection
    "GDPR",                # General Data Protection Regulation
    "Basel III",           # Regulatory framework for banks, including cybersecurity controls
    "FIPS 140-2",          # Federal Information Processing Standard for cryptography
    "EU NIS Directive",    # EU Network and Information Security Directive
    "Cybersecurity Maturity Model Certification (CMMC)", # Cybersecurity standard for defense contractors
    "SARBANES-OXLEY (SOX)",# Compliance standard for financial data protection
    "OECD Privacy Guidelines" # International guidelines for data privacy
]
common_threats_options = ["", "", "",
    "ransomware",
    "phishing",
    "malware",
    "DDoS attacks",
    "SQL injection",
    "zero-day vulnerabilities",
    "social engineering",
    "insider threats",
    "man-in-the-middle attacks",
    "password attacks",
    "supply chain attacks",
    "privilege escalation",
    "unauthorized access",
    "exploitation of IoT devices",
    "data breaches",
    "advanced persistent threats (APTs)",
    "cross-site scripting (XSS)",
    "brute force attacks",
    "exfiltration of sensitive data"
]
data_sensitivity_options = ["", "", "", "general", "confidential", "financial", "healthcare", "intellectual property"]
industry_standard_options = ["", "", "", "ISO 27001", "NIST",
    "ISO 27001",         # Information security management
    "NIST Cybersecurity Framework",  # Comprehensive cybersecurity framework
    "PCI DSS",           # Payment Card Industry Data Security Standard
    "HIPAA",              # Health Insurance Portability and Accountability Act
    "CIS Controls",      # Center for Internet Security's best practices
    "GDPR",              # General Data Protection Regulation
    "COBIT",             # Control Objectives for Information and Related Technologies
    "FISMA",             # Federal Information Security Management Act
    "SOX",               # Sarbanes-Oxley Act for data protection in financials
    "ISO 27018",         # Protection of personal data in the cloud
    "CSA CCM",           # Cloud Security Alliance's Cloud Controls Matrix
    "SANS Critical Security Controls",  # Recommended cybersecurity practices
    "SOC 2",             # Service Organization Control reports for service providers
    "CMMC",              # Cybersecurity Maturity Model Certification
    "ITIL",              # IT service management framework (with security practices)
    "FIPS 140-2",        # Federal Information Processing Standard for cryptography
    "Basel III",         # Regulatory framework for banks with cybersecurity components
    "GDPR",              # General Data Protection Regulation for data privacy
    "OECD Privacy Guidelines"  # International data privacy standards
]
technology_adoption_options = ["","",""
    "IoT (Internet of Things)",                 # Connected devices and their security considerations
    "AI (Artificial Intelligence)",             # Advanced AI systems and their implications for cybersecurity
    "cloud services (public, private, hybrid)", # Cloud infrastructure and associated cybersecurity needs
    "blockchain",                               # Distributed ledger technology and its unique security challenges
    "big data",                                 # Handling massive datasets and ensuring data privacy and protection
    "machine learning",                         # ML algorithms and their security risks, such as adversarial attacks
    "edge computing",                           # Decentralized processing at the data source and its security concerns
    "5G networks",                              # The rollout of 5G and the potential risks it brings to connected devices
    "virtualization",                           # Virtual machines and container technology (e.g., Docker, Kubernetes)
    "quantum computing",                        # Emerging quantum technology and its impact on cryptography
    "mobile technology",                        # Mobile devices and mobile apps, including BYOD policies
    "remote work technologies",                 # Collaboration tools and remote access solutions
    "wearable technology",                      # Wearable devices and their potential data privacy issues
    "automated systems",                        # Robotic process automation (RPA) and its cybersecurity aspects
    "smart cities",                             # Urban IoT infrastructure and associated cybersecurity measures
    "cyber-physical systems",                   # Integration of physical processes with digital control systems
    "edge AI",                                  # Deploying AI models on edge devices and their security considerations
    "data lakes",                               # Centralized repositories for raw data and their protection needs
    "autonomous vehicles",                      # Self-driving cars and the cybersecurity requirements for vehicle networks
    "virtual reality (VR) and augmented reality (AR)", # Emerging technologies in entertainment and training
    "digital twins",                            # Simulated models of physical objects and their cybersecurity aspects
    "serverless computing",                     # Serverless architectures and their impact on security policies
    "secure software development practices",    # Emphasis on secure coding and SDLC methodologies
    "privacy-enhancing technologies (PETs)",     # Tech designed to protect user data and privacy
]

def generate_random_user_prompt():
    geographic_presence = random.choice(geographical_presence_options)
    company_size = random.choice(company_size_options)
    industry_sector = random.choice(industry_sector_options)
    third_party_dependency = random.choice(third_party_dependency_options)
    regulations = random.choice(regulations_options)
    common_threats = random.choice(common_threats_options)
    data_sensitivity = random.choice(data_sensitivity_options)
    industry_standard = random.choice(industry_standard_options)
    technology_adoption = random.choice(technology_adoption_options)

    user_prompt = "Generate a cybersecurity policy"
    if company_size != "":
        user_prompt += f" for a {company_size} company"
    else:
        user_prompt += " for a company"

    if geographic_presence != "":
        user_prompt += f" located in {geographic_presence }"

    if industry_sector != "":
        user_prompt += f" in the {industry_sector} sector"

    if third_party_dependency != "":
        user_prompt += f" with {third_party_dependency}"

    if regulations != "":
        user_prompt += f" subject to {regulations} regulations"

    if common_threats != "":
        user_prompt += f" and has common {common_threats} threats"

    if data_sensitivity != "":
        user_prompt += f" and uses {data_sensitivity} data"

    if industry_standard != "":
        user_prompt += f" following the {industry_standard} standard"

    if technology_adoption != "":
        user_prompt += f" leveraging {technology_adoption}"

    return user_prompt

In [ ]:
# ==== Generate 4 completions per prompt (RAG + HF chat) and save one row per prompt ====
# pip install -U huggingface_hub sentence-transformers pandas
import os, json, time, hashlib
import pandas as pd
from huggingface_hub import InferenceClient
from sentence_transformers import SentenceTransformer
import weaviate.classes as wvc
import random
from tqdm import tqdm # Changed from import tqdm

# -------- Config --------
HF_MODEL    = "meta-llama/Llama-3.2-1B-Instruct"    # instruct model
HF_PROVIDER = "novita"                              # or None for default HF Inference
MAX_TOKENS  = 700                                   # per completion
TOP_K_RAG   = 20                                    # how many chunks to fetch for RAG
OUT_JSONL   = "cyber_policies_4comps.jsonl"         # rich rows (includes lists)
OUT_CSV     = "cyber_policies_4comps.csv"           # flat view (4 comps as columns)

# diversity settings for the 4 completions
TEMPS  = (0.7, 0.9, 1.1, 1.2)
TOP_PS = (0.95, 0.92, 0.90, 0.85)

# -------- Helpers --------
def det_id(text: str) -> str:
    return hashlib.md5(text.encode("utf-8")).hexdigest()[:12]

def build_user_prompt(user_query: str, similar_chunks: list[dict]) -> str:
    # Keep it compact: include only text + a hint of source/section
    lines = []
    for ch in similar_chunks:
        src = ch.get("source") or "-"
        sec = ch.get("section") or "-"
        lines.append(f"[{src} | {sec}] {ch['text']}")
    context = "\n".join(lines)
    return f"""# User Query
{user_query}

# Retrieved Policy Chunks (RAG)
{context}
"""

def generate_4_chat_completions(hf_client: InferenceClient, system_prompt: str, user_prompt: str) -> list[dict]:
    base_msgs = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]
    outs, seen = [], set()
    for i in range(4):
        t, tp = TEMPS[i % len(TEMPS)], TOP_PS[i % len(TOP_PS)]
        attempt = 0
        while attempt < 5:  # Retry up to 5 times
            try:
                resp = hf_client.chat_completion(
                    messages=base_msgs,
                    max_tokens=MAX_TOKENS,
                    temperature=t,
                    top_p=tp
                )
                text = resp.choices[0].message["content"].strip()
                sig = " ".join(text.split())[:300]
                if sig in seen:
                    # nudge for diversity if duplicate
                    t = min(1.35, t + 0.1)
                    tp = max(0.80, tp - 0.05)
                    resp = hf_client.chat_completion(
                        messages=base_msgs,
                        max_tokens=MAX_TOKENS,
                        temperature=t,
                        top_p=tp
                    )
                    text = resp.choices[0].message["content"].strip()
                    sig = " ".join(text.split())[:300]
                seen.add(sig)
                outs.append({"text": text, "params": {"temperature": t, "top_p": tp}})
                time.sleep(30)  # Increased sleep time to avoid rate limiting
                break  # Break out of the retry loop if successful
            except Exception as e:
                attempt += 1
                wait_time = min(60, (2 ** attempt) + random.uniform(0, 5)) # Exponential backoff with jitter
                print(f"Attempt {attempt} failed: {e}. Retrying in {wait_time:.2f} seconds...")
                time.sleep(wait_time)
        if attempt == 5:
            print("Failed to generate completion after multiple retries.")
            outs.append({"text": "Error: Could not generate completion.", "params": {"temperature": t, "top_p": tp}}) # Append an error message
    return outs

# -------- One-time setup (faster than reloading each loop) --------
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
hf_client = InferenceClient(model=HF_MODEL, provider=HF_PROVIDER) if HF_PROVIDER else InferenceClient(model=HF_MODEL)

# -------- Main loop --------
rows_for_csv = []
total_time_elapsed = 0.0
entry_times = []

# open JSONL for streaming writes (safer if long runs)
with open(OUT_JSONL, "w", encoding="utf-8") as fjsonl:
    for i in tqdm(range(DATASET_SIZE)):
        formatted_time = time.strftime('%H:%M:%S', time.gmtime(total_time_elapsed))
        avg = (sum(entry_times) / len(entry_times)) if entry_times else 0
        average_time = time.strftime('%H:%M:%S', time.gmtime(avg))
        print(f"{round((i/DATASET_SIZE)*100.0, 2):.2f}% - {i+1}/{DATASET_SIZE} - {average_time}/iteration - {formatted_time}", end="\r")

        t0 = time.time()

        # ---- 1) prompt ----
        user_query = generate_random_user_prompt()

        # ---- 2) RAG: embed + vector search (self_provided vectors) ----
        q_vec = embedder.encode([user_query], normalize_embeddings=True)[0].tolist()
        resp = col.query.near_vector(
            near_vector=q_vec,
            limit=TOP_K_RAG,
            return_properties=["text", "source", "section"],
            return_metadata=wvc.query.MetadataQuery(distance=True),
        )
        similar_policy_chunks = [
            {"text": o.properties["text"],
             "source": o.properties.get("source"),
             "section": o.properties.get("section"),
             "distance": o.metadata.distance}
            for o in resp.objects
        ]

        user_prompt = build_user_prompt(user_query, similar_policy_chunks)

        # ---- 3) generate 4 completions ----
        completions = generate_4_chat_completions(hf_client, system_prompt, user_prompt)

        # ---- 4) save one row (JSONL) + flat CSV view ----
        row = {
            "prompt_id": det_id(user_query),
            "prompt": user_query,
            "rag_topk": TOP_K_RAG,
            "rag_chunks": similar_policy_chunks,   # rich context for debugging/scoring
            "completions": completions,            # list of 4 dicts: text + params
            "scores": None,                        # to be filled later by GPT-4
            "ranked_indices": None,
            "ranked_scores": None
        }
        fjsonl.write(json.dumps(row, ensure_ascii=False) + "\n")

        # flat CSV row (only texts)
        rows_for_csv.append({
            "prompt": user_query,
            "comp_0": completions[0]["text"],
            "comp_1": completions[1]["text"],
            "comp_2": completions[2]["text"],
            "comp_3": completions[3]["text"],
        })

        # progress timing
        dt = time.time() - t0
        entry_times.append(dt)
        total_time_elapsed += dt

# final progress line
formatted_time = time.strftime('%H:%M:%S', time.gmtime(total_time_elapsed))
avg = (sum(entry_times) / len(entry_times)) if entry_times else 0
average_time = time.strftime('%H:%M:%S', time.gmtime(avg))
print(f"{round(((i+1)/DATASET_SIZE)*100.0, 2):.2f}% - {i+1}/{DATASET_SIZE} - {average_time}/iteration - {formatted_time}")

# write compact CSV
pd.DataFrame(rows_for_csv).to_csv(OUT_CSV, index=False)
print(f"\nWrote rows → {OUT_JSONL} and {OUT_CSV}")

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 1/300 [04:30<22:26:40, 270.24s/it]

  1%|          | 2/300 [09:02<22:28:12, 271.45s/it]

  1%|          | 3/300 [14:04<23:31:44, 285.20s/it]

  1%|▏         | 4/300 [18:41<23:11:19, 282.03s/it]

...

 15%|█▍        | 44/300 [3:25:31<18:57:32, 266.61s/it]

 15%|█▌        | 45/300 [3:29:55<18:50:01, 265.89s/it]

 15%|█▌        | 46/300 [3:35:33<20:17:32, 287.61s/it]

 15%|█▌        | 46/300 [3:37:06<19:58:50, 283.19s/it]


KeyboardInterrupt: 

In [10]:
df_test = pd.read_csv("cyber_policies_4comps.csv")
df_test.head()

FileNotFoundError: [Errno 2] No such file or directory: 'cyber_policies_4comps.csv'

In [ ]:
!pip install protobuf==3.20.3
!pip install grpcio==1.59.5